# 40 — Validação e índice

Verifica os Parquets gerados, registra número de linhas, colunas, tamanho e SHA-256 e produz um índice local. Não pressupõe quantidade fixa de municípios nem período fixo: trabalha com o conteúdo efetivamente produzido.


In [ ]:
%pip -q install pandas pyarrow


In [ ]:
from pathlib import Path
import hashlib, re
import pandas as pd
import pyarrow.parquet as pq
ROOT=Path.cwd(); OUT=ROOT/"dados"/"processado"; CONTROL=ROOT/"dados"/"controle"; CONTROL.mkdir(parents=True,exist_ok=True)
def sha256(p,chunk=1024*1024):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()
rows=[]
for p in sorted(OUT.rglob("*.parquet")):
    meta=pq.ParquetFile(p).metadata
    rows.append({"arquivo":str(p.relative_to(ROOT)),"nome":p.name,"bytes":p.stat().st_size,"linhas":meta.num_rows,"row_groups":meta.num_row_groups,"colunas":meta.num_columns,"sha256":sha256(p)})
idx=pd.DataFrame(rows)
out=CONTROL/"indice_particoes.csv"; idx.to_csv(out,index=False)
print("Partições indexadas:",len(idx)); print("Linhas acumuladas:",int(idx.linhas.sum()) if len(idx) else 0); print("Salvo:",out)
display(idx.head(20))
